# 28 - Reading the star mask: the green miss was the measurement

**Purpose.** Explain what notebook `27` found when it measured the sky-pair noise again with the
stars masked out, and what that changes, for someone deciding what to do next. `27` is written for
someone *checking* it; this one is slow on purpose.

It walks through four things: what the green miss was, why stars were the suspect, what the mask
did to each cell, and the verdict table before and after. It ends on what is still open.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `mask_constants.json` and `masked_pairs.csv`, and `25`'s `stack_pairs.csv` and
`stack_constants.json` beside them. If anything here disagrees with `results/`, `results/` is right
and this notebook is the bug. It does not re-decide the mask either: the mask, its growth and the
publishing rule were fixed in `27`'s purpose cell before any noise was seen.

**It assumes** `26_contract3_read.ipynb` for the stacks, the pairs and the SNR estimator, and
`00_statistics.ipynb` for why a spread over thousands of blocks is sharp.

**The headline, recomputed in section 1 rather than typed here:** with the stars masked, all four
green pairs pass and the green miss has no direction left. MISSION's last untested assumption
survives this night.

In [ ]:
import json
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M

RESULTS = pathlib.Path.cwd().parent / "results"
masked = M.Constants.load(RESULTS / "mask_constants.json")
stack = M.Constants.load(RESULTS / "stack_constants.json")
pairs = pd.read_csv(RESULTS / "masked_pairs.csv")
published = pd.read_csv(RESULTS / "stack_pairs.csv")
sky = json.loads((RESULTS / "sky_constants.json").read_text())
model_eta = json.loads((RESULTS / "model_constants.json").read_text())["eta_comb"]

CELLS = {"A": "gain 50, 30 s", "B": "gain 50, 120 s", "C": "gain 200, 30 s", "D": "gain 200, 120 s"}
PLANE_SET = {"green": ["G1", "G2"], "B": ["B"]}
MASKS = ["none", "grow0", "grow1", "grow2", "tight"]
VERDICT_MASK = "tight"

## 1. The verdict, before and after

MISSION's test: rank at least three pairs right, to within 10%, one of them straddling the HCG
threshold at gain 200, and each one a pair the model predicts *apart* by more than the SNR
measurement's own repeatability. `none` is `27`'s unmasked re-run, which reproduces `25`; `tight`
is the mask the verdict was fixed to before anything was measured.

In [ ]:
side = (pairs[pairs["mask"] == "none"]
        .merge(pairs[pairs["mask"] == VERDICT_MASK], on=["pair", "why", "plane", "predicted_pct"],
               suffixes=("_none", "_masked")))
cols = ["pair", "why", "plane", "predicted_pct", "measured_pct_none", "verdict_none",
        "measured_pct_masked", "repeatability_pct_masked", "verdict_masked"]
print(side[cols].round(2).to_string(index=False))

green = side[side.plane == "green"]
passed = green[green.verdict_masked == "pass"]
straddle = passed.why.str.contains("straddles").any()
print(f"\ngreen, masked: {len(passed)} of {len(green)} pass, "
      f"{(green.verdict_masked == 'tie').sum()} ties, a pass straddles HCG: {straddle}")
print("definition of done:", "MET" if len(passed) >= 3 and straddle else "NOT MET")
assert masked.entry("ranked_pairs_masked")["value"] == {
    f"{r.pair} {r.plane}": r.verdict_masked for r in side.itertuples()}, \
    "masked_pairs.csv and mask_constants.json disagree"

**The miss had a direction, and now it does not.** The next cell puts each miss in points and in
units of its own repeatability. Unmasked, every green pair came in *below* its prediction. A miss
that always has the same sign is a missing term or a biased measurement, never noise - which is
why `26` could not leave it alone.

In [ ]:
for tag in ("none", "masked"):
    side[f"miss_{tag}"] = side[f"measured_pct_{tag}"] - side.predicted_pct
    side[f"sigmas_{tag}"] = side[f"miss_{tag}"] / side[f"repeatability_pct_{tag}"]
print(side[["pair", "plane", "miss_none", "sigmas_none", "miss_masked", "sigmas_masked"]]
      .round(2).to_string(index=False))
for plane in PLANE_SET:
    s = side[side.plane == plane]
    print(f"\n{plane:5s} mean miss {s.miss_none.mean():+.2f} points unmasked, "
          f"{s.miss_masked.mean():+.2f} masked; signs masked: "
          f"{int((s.miss_masked > 0).sum())} over, {int((s.miss_masked < 0).sum())} under")

### What that does and does not say

**Green went from one-way to mixed.** Two pairs now land over their prediction and two under, and
the mean miss is near zero. Each one left is one to two times its repeatability, which is what a
model with nothing missing should leave.

**Blue's mean miss shrank too, but two blue pairs are not clean.** Blue's misses were already mixed
in sign before the mask, and its stars clip less, so it had less to lose. Under the mask, gain 50
from 30 to 120 s lands 9 points *over* its prediction, 4.8 times its repeatability, and gain 50 to
200 at 120 s lands 5.5 points under, 2.8 times. Both still pass, because the bar is "within 10% on
the ratio". But a miss of nearly five repeatabilities is not noise, and nothing in this notebook
explains it. It is named again in section 6.

**The pass bar is still loose.** As `26` said, "within 10%" is on the ratio, so a pass means right
direction and right rough size, not right to the measurement's precision. Section 1's second table
is the one to read for that.

## 2. Why stars were the suspect

`25` named two cheap suspects for the green miss, and `27` section 1 ruled both out by arithmetic
on published numbers: the dark bound is about 0.0005 e-/px/s against a sky near 1.9, and the
object's own light is about 6% of the sky, which moves no prediction by more than 2 points against
misses of 5-10.

What pointed at stars is this. Each pair's measured ratio over its predicted ratio says how far the
second cell fell short *relative to* the first. Set cell A to zero and chain the pairs, and each
cell gets a shortfall against the model. Beside it, the share of pixels that clip in each cell, the
plainest count of how bright the stars are in counts.

In [ ]:
def shortfall(mask):
    p = pairs[pairs["mask"] == mask]
    ratio = {(r.pair, r.plane): (1 + r.measured_pct / 100) / (1 + r.predicted_pct / 100)
             for r in p.itertuples()}
    out = {}
    for plane in PLANE_SET:
        s = {"A": 1.0, "B": ratio["A->B", plane], "C": ratio["A->C", plane]}
        s["D"] = s["C"] * ratio["C->D", plane]
        # B->D is implied by the other three, so it is a free check on the arithmetic
        assert abs(s["D"] / s["B"] / ratio["B->D", plane] - 1) < 1e-4
        out |= {(plane, c): 100 * (v - 1) for c, v in s.items()}
    return pd.Series(out)


clip = sky["clipped_fraction"]["value"]
cells = pd.DataFrame({
    "setting": [CELLS[c] for _, c in shortfall("none").index],
    "clipped_pct": [100 * np.mean([clip[c][p] for p in PLANE_SET[plane]])
                    for plane, c in shortfall("none").index],
    "shortfall_none": shortfall("none"),
    "shortfall_masked": shortfall(VERDICT_MASK)})
print(cells.round(3).to_string())

**Unmasked, the shortfall follows the clipping.** In green, A clips least and falls short by
nothing (it is the zero), then B and C, then D, which clips about four times more than C and falls
short by about twice as much. Blue clips less in every cell and falls short less.

**Why stars would do that.** `stats.diff_sigma` measures noise from the difference of two
half-stacks, and it already clips star *cores* at 5 sigma. What it cannot reach is the *wings* of a
star whose profile differs between the two halves - a little more seeing in one, or a clipped core
that the rejection handled differently. Those residues sit below the clip and widen the spread, and
the brighter the star in counts, the bigger they are. Longer subs and higher gain both make stars
brighter in counts, and both are what each pair moves towards. So the estimator overstated the
noise most in exactly the cells the model expected to win, and the model looked over-optimistic.

**Masked, the shortfall is gone.** The last column is the same chain under `tight`. No green cell
is more than 3% off, the signs are mixed, and the ordering by clipping has gone with it. In blue,
cell B now sits 6% *over* - the same gain 50, 30 to 120 s overshoot as section 1.

## 3. What the mask did to each cell

The signal is held fixed between the two runs - it is a median over the whole box, which stars
barely move, and `27` does not mask it. So when a pair's measured ratio changes under the mask,
only the noise moved. That lets the published pairs say how much the noise fell in each cell,
*relative to cell A*. (The fall in A itself is not in `results/`; `27` printed it and the decision
log records it, but a ratio of pairs cannot recover it.)

For green the ratio is over a mean of two planes, so this reading is close rather than exact.

In [ ]:
def noise_vs_A(mask):
    # (1 + masked) / (1 + none) for X->Y is (noise kept in X) / (noise kept in Y)
    p = pairs.pivot_table(index=["pair", "plane"], columns="mask", values="measured_pct")
    move = (1 + p[mask] / 100) / (1 + p["none"] / 100)
    out = {}
    for plane in PLANE_SET:
        r = {"A": 1.0, "B": 1 / move["A->B", plane], "C": 1 / move["A->C", plane]}
        r["D"] = r["C"] / move["C->D", plane]
        out |= {(plane, c): 100 * (v - 1) for c, v in r.items()}
    return pd.Series(out)


ladder = pd.DataFrame({m: noise_vs_A(m) for m in MASKS[1:]})
print("noise change under each mask, relative to cell A's own change, %:")
print(ladder.round(2).to_string())

**The noise fell most in D, and by about the size of D's shortfall.** In green, D's noise fell by
roughly 13% more than A's, and D's unmasked shortfall in section 2 was the same. B and C land in
between, but in swapped order: B's noise fell more than C's, while B's shortfall was the smaller.
Their clip shares are close. If the miss had been a missing term in the model, the mask would have
taken the same fraction off every cell and this table would be zeros.

**It levels off at once.** The four columns run from the bare star pixels (`grow0`) to stars grown
by three pixels (`tight`). Most rows agree to within a point; the widest, blue C, spans under three. `27` set this ladder up
expecting the noise to keep falling as the mask grew and then level off before `tight`. It was
already level at `grow0`. So the excess sat on the star pixels themselves and their nearest
neighbours, not far out in faint wings. The word *wings* in section 2 overstates how far out it
reached.

## 4. What the mask costs: the SNR of the gaps between stars

NGC 7000 sits in the Milky Way, so there are stars nearly everywhere in the signal box. Every 4x4
block that touches a masked pixel is left out of the noise.

In [ ]:
mask = masked.entry("star_mask")
TOTAL = (512 // 4) ** 2           # the signal ROI is 512 x 512 plane pixels, binned 4x4
cover = pd.DataFrame(mask["value"]).T
cover["blocks_kept"] = (TOTAL * (1 - cover.block_pct / 100)).round().astype(int)
print(cover.to_string())
print(f"\n{mask['note']}")

**`tight` measures the noise on about 15% of the box.** That is the darkest part of it: the gaps
between stars. It is still about 2400 blocks, enough for a sharp spread (`00`).

**Fair for ranking; not the SNR of the field.** The mask is one mask, the union over all sixteen
full stacks, so every cell loses the same patch of sky and a ranking compares like with like. That
is why it was built that way rather than one mask per cell. But the signal is a median of the whole
box and the noise comes from its gaps, so a masked SNR is not "the SNR of NGC 7000 at these
settings", and it must not be quoted as one. MISSION only asks for a ranking, so the verdict does
not need it to be.

## 5. The two older constants still carry the unmasked noise

`25` published two other numbers from the same estimator, with no mask. Neither one moves the
verdict above, and this section shows why.

In [ ]:
rep = stack.entry("snr_repeatability")
print("snr_repeatability (25, unmasked), % per cell:", rep["value"])
r = side[["pair", "plane", "predicted_pct", "repeatability_pct_none", "repeatability_pct_masked"]].copy()
r["margin_masked"] = r.predicted_pct.abs() / r.repeatability_pct_masked
print(r.round(2).to_string(index=False))
print(f"\nclosest pair to a tie, masked: {r.loc[r.margin_masked.idxmin(), 'pair']} "
      f"{r.loc[r.margin_masked.idxmin(), 'plane']}, predicted apart by "
      f"{r.margin_masked.min():.1f} times its repeatability")

**`snr_repeatability` did not decide anything here.** `27` measured each pair's repeatability
again under each mask, and the masked verdicts use that, not `25`'s. It is the
`repeatability_pct` column of `masked_pairs.csv`. Every pair is predicted apart by at least 3.5
times it, masked or not, so no pair is a tie whichever one you read. The unmasked per-cell figure in `stack_constants.json` stays as
`25` published it, a statement about the unmasked estimator.

In [ ]:
reg = stack.entry("eta_comb_registered")
print("eta_comb_registered (25, unmasked, registered lights):", reg["value"])
print("eta_comb the model reads:", model_eta["value"])
print(f"  from {model_eta['notebook']}: {model_eta['note'][:120]}...")

**`eta_comb_registered` is not read by the model.** The model's `eta_comb` is still session 03's
bias ladder, carried in `model_constants.json`. The registered ladder is the better number for real
dithered lights - `26` section 3 says why - but nothing has switched the model over to it. So the
unmasked residue in it moves no prediction today. Re-measuring it under the mask would pin down no
term the model uses, and that is the rule for not running a measurement.

**It becomes due the day someone proposes the switch.** At that point it has to be measured under
the mask first, and the switch also needs a ladder that reaches N of hundreds, which this night
cannot give.

## 6. What is settled, and what is next

**Settled.**
- The one-way green miss that `25` found was the SNR estimator, not the model. Star residues inflated
  the noise most in the cells with the brightest stars, which are the cells each pair moves towards.
- With one fixed star mask on every cell, all four green pairs and all four blue pairs pass, none is
  a tie, and two of them straddle HCG. MISSION's definition of done is met with room to spare.
- MISSION's last untested assumption - no noise term that fails to shrink with `t` - survives this
  night. Nothing here asks for one.

**Not settled, named rather than fixed.**
1. **One night.** The assumption survives this night, at -20 C, on one target. That is a pass, not
   a proof.
2. **Blue, gain 50, 30 to 120 s, overshoots by 4.8 repeatabilities.** It passes, and it is the
   opposite sign to the old green miss, so it is not the same problem. It is the largest miss left
   in units of its own precision, and nothing here explains it.
3. **The model's `eta_comb` is still the bias ladder.** Switching to registered lights is a model
   decision. It needs `eta_comb_registered` measured under the mask, and a ladder that reaches N of
   hundreds.
4. **Star clipping per star**, the star-colour half of the Pareto curve. The mask is a step towards
   it, but it needs per-star photometry, and a notebook with its own agreed purpose.